# Corpus Statistics
Word-level statistics for the reference transcripts, the Dialect-Aware Transcription (DAT),
and the Dialect-Ignorant Transcription (DIT). All text is cleaned with the shared `clean()` helper
(lowercase, ß→ss, umlauts retained, non-German characters stripped).

## Setup

In [59]:
import sys
import importlib
from pathlib import Path
from collections import Counter

import pandas as pd
import math

sys.path.insert(0, str(Path('__file__').resolve().parent))
import utils; importlib.reload(utils)
from utils import clean

In [60]:
PROJECT_ROOT = Path('__file__').resolve().parent.parent
ANALYSIS_DIR = PROJECT_ROOT / '02-analysis-position'
DAT_TSV = ANALYSIS_DIR / 'dialect-aware-transcript.tsv'
DIT_TSV = ANALYSIS_DIR / 'dialect-ignorant-transcript.tsv'

## Load Data

In [61]:
df_dit = pd.read_csv(DIT_TSV, sep='\t', encoding='utf-8-sig')
df_dat = pd.read_csv(DAT_TSV, sep='\t', encoding='utf-8-sig')
df = pd.merge(df_dit, df_dat[['path', 'DAT']], on='path', how='inner')

print(f"Clips: {len(df)}")
df.head(3)

Clips: 100


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender,DIT,DAT
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,4.778662,Insgesamt habe ich einige Hunderttausend Frank...,news,300bb931-79ae-40ec-b989-3efd5e83f4c2,Zürich,ZH,8408.0,fourties,male,Insgesamt habe ich einige hundert Dusik vor un...,Insgesamt habe ich einige 100.000 Franken verl...
1,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dca...,5.294150,Welche Rolle hatte Rosenberg während des Holoc...,news,6e084270-8d26-43d9-ba69-5e8ee793ab8c,Zürich,ZG,6340.0,twenties,female,"Weli Rolle, höchter Uz stoodhergewerden von Ho...",Wer ironne hättet viele bands auf den einzugem...
2,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,5.802676,Das ist angesichts aller schlechten Optionen d...,news,6858a37b-edd0-4fdf-871c-96d3b1bd3e21,Zürich,ZH,8704.0,fourties,male,Das ist angesichts vor allen Schlachten Option...,Das ist angesichts von allen schlechten Option...


## flatten words into single list (and clean data)

In [62]:
def tokenise(series):
    """Return (tokens, per_clip_sets) for a pandas Series of sentences."""
    tokens = []
    per_clip = []
    for text in series.dropna().astype(str):
        words = [clean(w) for w in text.split()]
        words = [w for w in words if w]  # drop empty strings after cleaning
        tokens.extend(words)
        per_clip.append(set(words))
    return tokens, per_clip

src_tokens,  src_clips  = tokenise(df['sentence'])
dit_tokens,  dit_clips  = tokenise(df['DIT'])
dat_tokens,  dat_clips  = tokenise(df['DAT'])

## Most Frequent Words

In [63]:
TOP_N = 20

def freq_table(tokens, label, n=TOP_N):
    counts = Counter(tokens)
    total = len(tokens)
    rows = counts.most_common(n)
    df = pd.DataFrame(rows, columns=['word', 'count']).assign(source=label)
    df['ratio'] = (df['count'] / total).round(4)
    return df

def least_freq_table(tokens, label, n=TOP_N):
    counts = Counter(tokens)
    total = len(tokens)
    rows = counts.most_common()[:-n-1:-1]  # last n in ascending order
    df = pd.DataFrame(rows, columns=['word', 'count']).assign(source=label)
    df['ratio'] = (df['count'] / total).round(4)
    return df

top = pd.concat([
    freq_table(src_tokens, 'Source'),
    freq_table(dit_tokens, 'DIT'),
    freq_table(dat_tokens, 'DAT'),
], ignore_index=True)

bottom = pd.concat([
    least_freq_table(src_tokens, 'Source'),
    least_freq_table(dit_tokens, 'DIT'),
    least_freq_table(dat_tokens, 'DAT'),
], ignore_index=True)

SOURCE_ORDER = ['Source', 'DIT', 'DAT']
top['source'] = pd.Categorical(top['source'], categories=SOURCE_ORDER, ordered=True)
bottom['source'] = pd.Categorical(bottom['source'], categories=SOURCE_ORDER, ordered=True)

print(f"Top {TOP_N} most frequent words per corpus:")
display(top.pivot_table(index=top.groupby('source', observed=True).cumcount(), columns='source', values=['word', 'count', 'ratio'], aggfunc='first', observed=True))

print(f"\nTop {TOP_N} least frequent words per corpus:")
display(bottom.pivot_table(index=bottom.groupby('source', observed=True).cumcount(), columns='source', values=['word', 'count', 'ratio'], aggfunc='first', observed=True))

Top 20 most frequent words per corpus:


count           ratio                   word              
source Source DIT DAT  Source     DIT     DAT Source    DIT    DAT
0          33  32  34  0.0393  0.0360  0.0403    die    die    die
1          15  19  26  0.0179  0.0213  0.0308    das    das    der
2          14  17  15  0.0167  0.0191  0.0178   eine    der    und
3          13  16  15  0.0155  0.0180  0.0178    und    ist    mit
4          12  14  13  0.0143  0.0157  0.0154    ist    ich    ist
5          11  12  13  0.0131  0.0135  0.0154    der    und    ein
6          10  12  12  0.0119  0.0135  0.0142  nicht    ein    das
7          10  11  11  0.0119  0.0124  0.0130     in     in     in
8           9  11   9  0.0107  0.0124  0.0107    mit    wir    hat
9           9  11   8  0.0107  0.0124  0.0095    den     er    auf
10          8  10   8  0.0095  0.0112  0.0095    ich    hat     zu
11          8  10   8  0.0095  0.0112  0.0095     er     es    für
12          8  10   7  0.0095  0.0112  0.0083     zu    auf    ich
13          7   9   7  0.0083  0.0101  0.0083     es    für    von
14          7   8   7  0.0083  0.0090  0.0083   sich    mit    wie
15          7   8   7  0.0083  0.0090  0.0083     im    dem   sich
16          7   8   7  0.0083  0.0090  0.0083    auf  nicht     er
17          6   8   6  0.0071  0.0090  0.0071   auch     da  nicht
18          6   7   6  0.0071  0.0079  0.0071    ein    von     es
19          6   6   6  0.0071  0.0067  0.0071    bei   sich   sind


Top 20 least frequent words per corpus:


count           ratio                            word                  \
source Source DIT DAT  Source     DIT     DAT          Source             DIT   
0           1   1   1  0.0012  0.0011  0.0012          sicher          sicher   
1           1   1   1  0.0012  0.0011  0.0012          gesagt          gesagt   
2           1   1   1  0.0012  0.0011  0.0012         ehrlich         ehrlich   
3           1   1   1  0.0012  0.0011  0.0012              da       berachnet   
4           1   1   1  0.0012  0.0011  0.0012       berechnet           schaf   
5           1   1   1  0.0012  0.0011  0.0012        geschäft            satz   
6           1   1   1  0.0012  0.0011  0.0012  stundenansätze          stunde   
7           1   1   1  0.0012  0.0011  0.0012       kriterien  wartekritärien   
8           1   1   1  0.0012  0.0011  0.0012         welchen       bezwiegst   
9           1   1   1  0.0012  0.0011  0.0012             bzw         wiegalt   
10          1   1   1  0.0012  0.0011  0.0012        heiraten     kortsparakt   
11          1   1   1  0.0012  0.0011  0.0012            bald            sper   
12          1   1   1  0.0012  0.0011  0.0012           sogar        schweppe   
13          1   1   1  0.0012  0.0011  0.0012            will   evticaroblaam   
14          1   1   1  0.0012  0.0011  0.0012      liebespaar        nichtrad   
15          1   1   1  0.0012  0.0011  0.0012           leben       hollywood   
16          1   1   1  0.0012  0.0011  0.0012           darum           umzug   
17          1   1   1  0.0012  0.0011  0.0012            gehe           küche   
18          1   1   1  0.0012  0.0011  0.0012          bereit            gute   
19          1   1   1  0.0012  0.0011  0.0012       hollywood           freue   

                       
source            DAT  
0             züchern  
1              gesagt  
2             ehrlich  
3           berechnet  
4            geschäft  
5                satz  
6              stunde  
7              werden  
8            kriterie  
9             welcher  
10      bezwiegezwies  
11       faterinsätze  
12            techlat  
13            358rsch  
14              grote  
15               blab  
16            drunter  
17                 ja  
18              parat  
19          hollywood

## Specific words
Words that occur only one time in the corpus. A high percentage indicates a broad vocabulary.

In [64]:
def words_used_once(tokens, label):
    counts = Counter(tokens)
    rare = sorted(w for w, c in counts.items() if c == 1)
    print(f"{label}: {len(rare):,} words appear only once ({100*len(rare)/len(counts):.1f}% of all unique words)")
    return rare

src_hapax = words_used_once(src_tokens, 'Source')
dit_hapax = words_used_once(dit_tokens, 'DIT')
dat_hapax = words_used_once(dat_tokens, 'DAT')

print("\nExamples (Source):", src_hapax[:15])

Source: 437 words appear only once (82.9% of all unique words)
DIT: 490 words appear only once (86.6% of all unique words)
DAT: 469 words appear only once (86.5% of all unique words)

Examples (Source): ['10', '100', '13', '2021', '2890', '30', '35', '36', '8380245', 'aargau', 'ab', 'abgesperrt', 'abtreibung', 'ac', 'alle']
